In [59]:
import pandas as pd

dataset = pd.read_csv("dataset/by_antibiotic_11comorbdims/CRO.csv")

In [60]:
print(dataset.columns)

Index(['Unnamed: 0', 'anon_id', 'order_time_jittered_utc_shifted', 'resistant',
       'adi_score', 'age', 'gender', 'nursing_home_visit_culture',
       'last_dose_to_culture', 'procedure_description',
       'procedure_days_culture', 'wards_int', 'Comorbidity_Reduced_Val_1',
       'Comorbidity_Reduced_Val_2', 'Comorbidity_Reduced_Val_3',
       'Comorbidity_Reduced_Val_4', 'Comorbidity_Reduced_Val_5',
       'Comorbidity_Reduced_Val_6', 'Comorbidity_Reduced_Val_7',
       'Comorbidity_Reduced_Val_8', 'Comorbidity_Reduced_Val_9',
       'Comorbidity_Reduced_Val_10', 'Comorbidity_Reduced_Val_11'],
      dtype='str')


In [47]:
AUTO_SAVE = True
MODEL_CODE = "CRO_hgbc_alpha0.01_maxiter5000_521features"
RESAMPLE = False
SAMPLES = 5000
OVERSAMPLE = False
UNDERSAMPLE = False

In [48]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVC

clf = make_pipeline(StandardScaler(), HistGradientBoostingClassifier(learning_rate=0.01, max_iter=5000,class_weight="balanced"))
# clf = make_pipeline(StandardScaler(), SVC(gamma="auto",class_weight="balanced"))

In [49]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.decomposition import PCA
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTENC
from sklearn.utils import resample
if UNDERSAMPLE:
    rs = RandomUnderSampler(sampling_strategy=1)

if OVERSAMPLE:
    cat_cols = list(dataset.drop(columns=['Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'anon_id','order_time_jittered_utc_shifted', 'resistant','adi_score','nursing_home_visit_culture', 'last_dose_to_culture','procedure_days_culture']).columns)
    print(cat_cols)
    rs = SMOTENC(categorical_features=cat_cols, sampling_strategy=1)


# pca = make_pipeline(
#     PCA(n_components=9)
# )

encoder = OrdinalEncoder()
X = dataset.drop(columns=["anon_id","order_time_jittered_utc_shifted","resistant",'Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0']).fillna(-9999999).astype(str)

# X = pca.fit_transform(X)

y = dataset["resistant"]



X = encoder.fit_transform(X)

# X = np.nan_to_num(X, -9999999)

y=y.to_numpy()


if OVERSAMPLE or UNDERSAMPLE:
    X_rs, y_rs = rs.fit_resample(X, y)
    y_rs = y_rs.to_numpy()
    X_rs = encoder.fit_transform(X_rs)

if RESAMPLE:
    X, y = resample(X, y, n_samples=SAMPLES, replace=False, stratify=y)
    if OVERSAMPLE or UNDERSAMPLE:
        X_rs, y_rs = resample(X_rs, y_rs, n_samples=SAMPLES, replace=False, stratify=y_rs)

if OVERSAMPLE or UNDERSAMPLE:
    X_train, X_test, y_train, y_test = train_test_split(X_rs, y_rs, test_size=0.2, stratify=y_rs)
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
X_train_us, X_test_us, y_train_us, y_test_us = train_test_split(X, y, test_size=0.2, stratify=y)



In [50]:
print(len(y_train_us))

15011


In [51]:
clf.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('histgradientboostingclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[bool](2,)","[False, True]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,515
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [52]:
if AUTO_SAVE:
    import pickle
    with open(f"CML Models/{MODEL_CODE}.model", "wb") as file:
        pickle.dump(clf, file)

In [53]:
# from sklearn.metrics import accuracy_score

# y_pred = clf.predict(X_test)
# print(accuracy_score(y_test, y_pred))


In [54]:
# print(clf.score(X_test,y_test))
# print(clf.score(X_train, y_train))

In [58]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

probs = clf.predict_proba(X_test_us)[:, 1]

y_pred = probs >= 0.5

tn, fp, fn, tp = confusion_matrix(y_test_us, y_pred).ravel().tolist()
print(f"    Accuracy: {accuracy_score(y_test_us, y_pred)}")
print(f"    Confusion Matrix\n     True negatives: {tn} ({tn / len(y_test_us)})\n     False positives: {fp} ({fp / len(y_test_us)})\n     False negatives: {fn} ({fn / len(y_test_us)})\n     True positives: {tp} ({tp / len(y_test_us)})")

from sklearn.metrics import roc_auc_score, average_precision_score
positive_probabilities = probs
auroc_score = roc_auc_score(y_test_us, positive_probabilities)
prauc_score = average_precision_score(y_test_us, positive_probabilities)
print(f"    AUROC: {auroc_score:.4f}")
print(f"    PRAUC: {prauc_score:.4f}")

    Accuracy: 0.7668531841193712
    Confusion Matrix
     True negatives: 2084 (0.555289102051692)
     False positives: 690 (0.1838529176658673)
     False negatives: 185 (0.049293898214761524)
     True positives: 794 (0.21156408206767918)
    AUROC: 0.8626
    PRAUC: 0.7128


In [56]:
print(sum(y_test) / len(y_test))

0.2608579802824407
